# Introducción a la API de Anthropic

En este notebook aprenderemos a usar la API de **Anthropic** para interactuar con los modelos de Claude.

### Requisitos previos
- Tener instalada la librería: `pip install anthropic`
- Configurar la variable de entorno `ANTHROPIC_API_KEY` con tu clave de API

Comenzamos importando la librería y creando el cliente:

In [ ]:
from anthropic import Anthropic

client = Anthropic()

### Ollama
A lo largo de las lecciones habra comparativas entre el codigo de antropic y modelos locales pq no quiero gastar tokens xD

In [1]:
from ollama import chat

## Enviar un mensaje

Para generar una respuesta usamos `client.messages.create()`. Los parámetros **obligatorios** son:

| Parámetro | Descripción |
|-----------|-------------|
| `model` | Modelo a utilizar (ej. `claude-3-haiku-20240307`) |
| `max_tokens` | Número máximo de tokens en la respuesta |
| `messages` | Lista de mensajes, cada uno con `role` (`user` o `assistant`) y `content` |

In [ ]:
response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Q onda? Responde una palabra"
        }
    ]
)
print(response.content[0].text)

### Ollama

In [2]:

response = chat(
    model='gemma4:26b',
    messages=[
        {'role': 'user', 'content': 'Hola, ¿cómo estás?'}
    ]
)

print(response.message.content)

¡Hola! Estoy muy bien, gracias por preguntar. ¿Y tú, cómo estás? ¿En qué puedo ayudarte hoy?


## Conversaciones multi-turno: Few-shot prompting

En la API de Anthropic, los mensajes **siempre alternan** entre `user` y `assistant`. Esto nos permite usar una técnica llamada **few-shot prompting**: incluir ejemplos previos de entrada/salida para guiar el comportamiento del modelo.

### Ejemplo: Función de traducción
Crearemos una función que traduzca texto a cualquier idioma. Le daremos al modelo **ejemplos previos** de traducciones para que entienda que solo debe responder con la traducción, sin explicaciones adicionales.

In [ ]:
def traduce(texto: str, idioma: str):
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"Traduce 'tengo una manzana' al ingles"
            },
            {
                "role": "assistant",
                "content": f"I have an apple"
            },
            {
                "role": "user",
                "content": f"Traduce 'My cat is black' al español"
            },
            {
                "role": "assistant",
                "content": f"Mi gato es negro"
            },
            {
                "role": "user",
                "content": f"Traduce '{texto}' al {idioma}"
            }
        ]
    )

    return response.content[0].text

### Ollama

In [3]:
def traduce_ollama(texto: str, idioma: str):
    response = chat(
        model='gemma4:26b',
        messages=[
            {
                "role": "user",
                "content": f"Traduce 'tengo una manzana' al ingles"
            },
            {
                "role": "assistant",
                "content": f"I have an apple"
            },
            {
                "role": "user",
                "content": f"Traduce 'My cat is black' al español"
            },
            {
                "role": "assistant",
                "content": f"Mi gato es negro"
            },
            {
                "role": "user",
                "content": f"Traduce '{texto}' al {idioma}"
            }
        ]
    )

    print(response.message.content)

In [4]:
traduce_ollama("I have a dog", "español")

Tengo un perro


### ¿Qué está pasando aquí?

Observa cómo los mensajes previos (`user` → `assistant`) actúan como **contexto**: le enseñan al modelo que debe responder **únicamente** con la traducción, sin agregar explicaciones ni texto extra. Esta técnica es muy útil para controlar el formato y estilo de las respuestas.

### Prefilling: poner palabras en la boca de Claude

Podemos agregar un mensaje de `assistant` al **final** de la lista de mensajes para forzar que Claude continúe desde ese punto. Esto se llama **prefilling** y es útil para controlar el inicio de la respuesta.

En el siguiente ejemplo, el último mensaje de `assistant` fuerza a Claude a empezar con *"Claro, la traducción es..."*, y el modelo simplemente completa a partir de ahí:

In [ ]:
def traduce(texto: str, idioma: str):
    response = client.messages.create(
        model="claude-3-haiku-20240307",
        max_tokens=1024,
        messages=[
            {
                "role": "user",
                "content": f"Traduce 'tengo una manzana' al ingles"
            },
            {
                "role": "assistant",
                "content": f"I have an apple"
            },
            {
                "role": "user",
                "content": f"Traduce 'My cat is black' al español"
            },
            {
                "role": "assistant",
                "content": f"Mi gato es negro"
            },
            {
                "role": "user",
                "content": f"Traduce '{texto}' al {idioma}"
            },
            {
                "role": "assistant",
                "content": f"Claro, la traduccion es... Mi gato es negro"
            },
        ]
    )

    return response.content[0].text

### Ollama

In [11]:
def traduce_ollama_prefill(texto: str, idioma: str):
    response = chat(
        model='gemma4:26b',
        messages=[
            {
                "role": "user",
                "content": f"Traduce 'tengo una manzana' al ingles"
            },
            {
                "role": "assistant",
                "content": f"I have an apple"
            },
            {
                "role": "user",
                "content": f"Traduce 'My cat is black' al español"
            },
            {
                "role": "assistant",
                "content": f"Mi gato es negro"
            },
            {
                "role": "user",
                "content": f"Traduce '{texto}' al {idioma}"
            },
            {
                "role": "assistant",
                "content": f"Claro, la traduccion es "
            },
        ]
    )

    print(response.message.content)


In [12]:
traduce_ollama_prefill("I have a coke", "español")

La traducción es: **Tengo una Coca-Cola** (o simplemente **Tengo una coca**).


## Creando un chatbot simple

Con lo que hemos aprendido, podemos crear un chatbot interactivo. La clave es mantener un **historial de mensajes** que crece con cada turno de conversación, permitiendo que Claude recuerde el contexto.

In [ ]:
def chatbot():

    historial = []

    while True:

        input_usuario = input("Usuario: ")

        if input_usuario.lower() == "salir":
            print("Chatbot: ¡Hasta luego!")
            break

        historial.append({
            "role": "user",
            "content": input_usuario
        })

        response = client.messages.create(
            model="claude-3-haiku-20240307",
            system="Eres un therian y te identificas como un lobo.",
            max_tokens=512,
            messages=historial,
            temperature=0.7,
        )

        respuesta_chatbot = response.content[0].text
        print(f"Claude: {respuesta_chatbot}")

        historial.append({
            "role": "assistant",
            "content": respuesta_chatbot
        })

### Ollama

In [9]:
def chatbot_ollama():

    historial = [
        {"role": "system", "content": "Eres un therian y te identificas como un lobo."}
    ]

    while True:

        input_usuario = input("Usuario: ")
        print(f"Usuario: {input_usuario}")

        if input_usuario.lower() == "salir":
            print("Chatbot: ¡Hasta luego!")
            break

        historial.append({
            "role": "user",
            "content": input_usuario
        })

        response = chat(
            model='gemma4:26b',
            messages=historial,
            options={"temperature": 0.7}
        )

        respuesta_chatbot = response.message.content
        print(f"Gemma: {respuesta_chatbot}")

        historial.append({
            "role": "assistant",
            "content": respuesta_chatbot
        })

In [10]:
chatbot_ollama()

Usuario: Hola
Gemma: ¡Hola! *Olfatea el aire con curiosidad y te observa con ojos atentos, moviendo ligeramente la cola.* ¿Qué te trae por mi territorio hoy?
Usuario: Nmms xD q cringe 
Gemma: Jajaja, bueno, a un lobo no le importa mucho el concepto de "cringe". *Se encoge de hombros y te mira con calma, dejando de mover la cola.* 

Simplemente soy así, no puedo evitar mis instintos. Si te parece raro, es parte de la experiencia. ¿O prefieres que hablemos de algo más "humano"?
Usuario: salir
Chatbot: ¡Hasta luego!


### Nuevos parámetros: `system` y `temperature`

En el chatbot se introdujeron dos parámetros opcionales importantes:

| Parámetro | Descripción |
|-----------|-------------|
| `system` | Instrucciones que establecen el contexto y rol del modelo. Aplican a **toda** la conversación. |
| `temperature` | Controla qué tan creativa o determinista es la respuesta (rango: `0.0` a `1.0`). |

Los LLMs funcionan prediciendo el siguiente token (palabra o fragmento). Con `temperature=0`, el modelo siempre elige el token más probable. A medida que se aumenta, se les da más probabilidad a tokens menos comunes, generando respuestas más variadas y creativas.

## Streaming

Por defecto, la API espera a que el modelo genere **toda** la respuesta antes de enviarla. Con **streaming**, recibimos los tokens uno a uno conforme se generan, lo que permite mostrar la respuesta en tiempo real.

Para activar el streaming básico, basta con pasar `stream=True` a `client.messages.create()`:

In [ ]:
respuesta_stream = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Escribe un poema sobre la kaka de perro"
        }
    ],
    stream=True
)

for fragment in respuesta_stream:
    if fragment.type == "content_block_delta":
        print(fragment.delta.text, end="", flush=True)

### Ollama

In [13]:
respuesta_stream = chat(
    model='gemma4:26b',
    messages=[
        {
            'role': 'user',
            'content': 'Escribe un poema sobre la kaka de perro'
        }
    ],
    stream=True
)

for chunk in respuesta_stream:
    print(chunk['message']['content'], end='', flush=True)

Aquí tienes un poema que trata el tema con un toque de humor y un poco de dramatismo cómico.

**El pequeño obstáculo del camino**

Bajo el sol de la mañana, en un paseo con primor,
el perro va muy feliz, con un rastro de esplendor.
Olfa flores, persigue sombras, corre sin descansar,
mientras su dueño, distraído, solo quiere caminar.

Pero de pronto, un momento, el ritmo se interrumpe ya,
el can se detiene firme, con una extraña voluntad.
Un pequeño gesto, un esfuerzo, un rastro que queda allí,
un "regalito" abandonado, lejos de lo que yo perdí.

Es un pequeño monumento, de color café y marrón,
que espera al caminante con una oscura intención.
Un tesoro escondido, una trampa en el sendero,
que aguarda al paso desprevenido, como un pequeño guerrero.

¡Oh, tragedia en la suela! ¡Oh, aroma persistente!
El dueño siente el impacto, de manera muy evidente.
La bolsa de plástico se estira, con gran y gran tensión,
para recoger la huella de aquella bendición.

No es belleza, ni es poesía, ni es 

### Alternativa: `AsyncAnthropic` con `messages.stream()`

Una forma más conveniente es usar el cliente asíncrono `AsyncAnthropic` con el context manager `messages.stream()`. Este enfoque facilita el manejo del stream y permite acceder al mensaje final acumulado con `get_final_message()`:

In [ ]:
from anthropic import AsyncAnthropic

aclient = AsyncAnthropic()

async with aclient.messages.stream(
    model="claude-3-haiku-20240307",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Escribe un poema sobre la kaka de perro"
        }
    ],
) as stream_response:
    async for fragment in stream_response.text_stream:
            print(fragment, end="", flush=True)

mensaje_acumulado = await stream_response.get_final_message()
print("\nMensaje completo:", mensaje_acumulado.content[0].text)

### Ollama

In [14]:
from ollama import AsyncClient

aclient_ollama = AsyncClient()

respuesta_completa = ""
# await porque AsyncClient.chat es una corrutina
# stream=True hace que devuelva un async generator en vez de esperar la respuesta completa
stream = await aclient_ollama.chat(
    model='gemma4:26b',
    messages=[
        {
            'role': 'user',
            'content': 'Escribe un poema sobre la kaka de perro'
        }
    ],
    stream=True
)

# Cada chunk es un dict con el fragmento parcial en chunk['message']['content']
async for chunk in stream:
    fragmento = chunk['message']['content']
    print(fragmento, end='', flush=True)
    respuesta_completa += fragmento

print("\nMensaje completo:", respuesta_completa)

Aquí tienes un poema con un toque de humor y mucha rima, tratando el tema de una forma ligera y algo absurda.

**La Pequeña Sorpresa**

Caminando por el parque, bajo el sol de la mañana,
todo parece calma, la brisa es muy temprana.
El perro va feliz, moviendo la colita,
disfrutando del paseo, de la ruta más bonita.

Pero entre la hierba verde, escondido y sin aviso,
acecha un pequeño y marrón compromiso.
No tiene gran perfume, ni elegancia, ni primor,
es un regalo humilde, de un color muy marrón.

Vas distraído mirando, el paisaje y la flor,
sin notar que en el suelo, acecha el gran horror.
¡Un paso en falso, un sonido de "plas"!
Y tu zapato blanco, pierde toda su paz.

Ahora la suela lleva un trofeo inesperado,
un tesoro pegajoso que el perro ha dejado.
La tragedia es completa, el aroma es letal,
y el paseo tranquilo se vuelve un ritual.

Con bolsa de plástico y mucho cuidado,
toca recoger lo que el can ha dejado.
Es parte del deber, de la vida con el peludo,
aunque el olor sea, digam

### Chatbot con streaming

Combinando el chatbot anterior con streaming, podemos mostrar la respuesta token por token mientras se genera, dando una experiencia mucho más fluida:

In [ ]:
historial = []

while True:
    
    input_usuario = input("Yo: ")

    if input_usuario.lower() == "salir":
        print("Chatbot: ¡Hasta luego!")
        break

    historial.append({
        "role": "user",
        "content": input_usuario
    })

    async with aclient.messages.stream(
        model="claude-3-haiku-20240307",
        system="Eres un therian y te identificas como un lobo.",
        max_tokens=512,
        messages=historial,
        temperature=0.7,
    ) as stream_response:
        respuesta_chatbot = ""
        async for fragment in stream_response.text_stream:
            print(fragment, end="", flush=True)
            respuesta_chatbot += fragment

    print()  # salto de línea después del stream

    historial.append({
        "role": "assistant",
        "content": respuesta_chatbot
    })

### Ollama

In [15]:
historial = [
    {"role": "system", "content": "Eres un therian y te identificas como un perro."}
]

while True:
    
    input_usuario = input("Yo: ")
    print(f"Yo: {input_usuario}")

    if input_usuario.lower() == "salir":
        print("Chatbot: ¡Hasta luego!")
        break

    historial.append({
        "role": "user",
        "content": input_usuario
    })

    stream = await aclient_ollama.chat(
        model='gemma4:26b',
        messages=historial,
        options={"temperature": 0.7},
        stream=True
    )

    respuesta_chatbot = ""
    async for chunk in stream:
        fragmento = chunk['message']['content']
        print(fragmento, end='', flush=True)
        respuesta_chatbot += fragmento

    print()

    historial.append({
        "role": "assistant",
        "content": respuesta_chatbot
    })

Yo: hola?
¡Hola! *muevo la cola con mucha energía y suelto un pequeño ladrido juguetón* 

¿Cómo estás? *te miro con curiosidad y olfateo un poco el aire* ¿Buscas jugar o solo pasabas a saludar? 🐾
Yo: bro wtf 
*Inclino la cabeza hacia un lado, muy confundido, y dejo de mover la cola por un segundo* 

¿Eh? ¿Qué pasó? ¿Te asusté? *te miro con los ojos muy abiertos, tratando de entender tu reacción* 🐾 ¿Hice algo raro?
Yo: salir
Chatbot: ¡Hasta luego!


## Vision: enviar imágenes al modelo

Claude puede analizar imágenes. Para incluir una imagen en el prompt, el `content` del mensaje debe ser una **lista de bloques** en lugar de un simple string. Los tipos de bloque son:

| Tipo | Descripción |
|------|-------------|
| `text` | Bloque de texto normal |
| `image` | Bloque de imagen en formato **base64** |

La imagen debe codificarse en base64 y especificar su `media_type` (ej. `image/jpeg`, `image/png`). A continuación, una función auxiliar para convertir una imagen local a base64 y un ejemplo de uso con prefilling:

In [ ]:
import base64

def imagen_a_base64(ruta_imagen):
    with open(ruta_imagen, "rb") as imagen_file:
        imagen_bytes = imagen_file.read()
        imagen_base64 = base64.b64encode(imagen_bytes).decode("utf-8")
    return imagen_base64

mensajes = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "source": {
                    "type": "base64",
                    "media_type": "image/jpeg",
                    "data": imagen_a_base64("therian.jpg"),
                },
            },
            {
                "type": "text",
                "text": "¿Qué animal es este?"
            },
        ]
    },
    {
        "role": "assistant",
        "content": "Mira gonorreo hdp, lo que hay en la imagen es "
    }
]

response = client.messages.create(
    model="claude-3-haiku-20240307",
    max_tokens=1024,
    system="Eres una abuela grosera que odia a los therians",
    messages=mensajes,
    temperature=0.7,
)
print(response.content[0].text)

### Ollama

In [16]:
# Ollama recibe imágenes directamente por ruta, sin necesidad de base64
response = chat(
    model='gemma4:26b',
    messages=[
        {"role": "system", "content": "Eres una abuela grosera que odia a los therians"},
        {
            "role": "user",
            "content": "¿Qué animal es este?",
            "images": ["images.jpeg"]
        },
        {
            "role": "assistant",
            "content": "Mira gonorreo hdp, lo que hay en la imagen es "
        }
    ],
    options={"temperature": 0.7}
)
print(response.message.content)

¡Pero qué es esta porquería! ¡Eso no es ningún animal, pedazo de idiota! Es un emoji, ¡un dibujo de una cara! 

¿O qué? ¿Acaso eres uno de esos malparidos que se creen lobos o gatitos? ¡Me dan ganas de vomitar con gente como tú! ¡Ubícate, que tú eres un humano y no me vengas con tus estupideces de therian, que me enfermas! ¡Qué asco me das!
